# Introducción
A la hora de usar lenguajes de programación para trabajar con datos, hay un verdadero superpoder: la capacidad de modificar y crear columnas sobre la marcha. En el machine learning, al proceso de crear nuevas columnas mediante las columnas originales en el conjunto de datos se le llama ingeniería de características. La ingeniería de características brinda nuevas soluciones en el análisis y modelado de datos, pues simplifica los problemas al hacer los datos más adecuados.

En este capítulo, volverás al dataset de ventas de videojuegos que viste antes en el sprint. Utilizando este dataset, aprenderás:

Varias formas de crear nuevas columnas a partir de los datos existentes.
Como escribir tus propias funciones.
Como usar el método apply() para crear columnas nuevas con base en criterios complejos de procesamiento que no se pueden cumplir con las funciones existentes de pandas.

Una de las habilidades más útiles en el análisis de datos es crear una nueva columna a partir de las columnas existentes del conjunto de datos para resolver problemas específicos. Vamos a ver el conjunto de datos de las ventas de videojuegos.

df = pd.read_csv('/datasets/vg_sales.csv')
print(df.head())

Una de las habilidades más útiles en el análisis de datos es crear una nueva columna a partir de las columnas existentes del conjunto de datos para resolver problemas específicos. Vamos a ver el conjunto de datos de las ventas de videojuegos.

import pandas as pd

df = pd.read_csv('/datasets/vg_sales.csv')
print(df.head())

                       name platform  year_of_release         genre publisher  \
0                Wii Sports      Wii           2006.0        Sports  Nintendo   
1         Super Mario Bros.      NES           1985.0      Platform  Nintendo   
2            Mario Kart Wii      Wii           2008.0        Racing  Nintendo   
3         Wii Sports Resort      Wii           2009.0        Sports  Nintendo   
4  Pokemon Red/Pokemon Blue       GB           1996.0  Role-Playing  Nintendo   

  developer  na_sales  eu_sales  jp_sales  critic_score  user_score  
0  Nintendo     41.36     28.96      3.77          76.0         8.0  
1       NaN     29.08      3.58      6.81           NaN         NaN  
2  Nintendo     15.68     12.76      3.79          82.0         8.3  
3  Nintendo     15.61     10.93      3.28          80.0         8.0  
4       NaN     11.27      8.89     10.22           NaN 

## Transformaciones de columnas mediante operadores aritméticos
Fíjate que el DataFrame de arriba incluye las ventas de tres regiones: NA (Norteamérica), EU (Europa) y Japón (JPN). Para crear una columna llamada 'total_sales', tenemos que generarla a partir de las otras columnas. Afortunadamente, es fácil de hacer:

df = pd.read_csv('/datasets/vg_sales.csv')

df['total_sales'] = df['na_sales'] + df['eu_sales'] + df['jp_sales']
print(df['total_sales'].head())

0    74.09
1    39.47
2    32.23
3    29.82
4    30.38
Name: total_sales, dtype: float64


Esto funciona porque la mayoría de las funciones matemáticas trabajan de forma vectorial: se aplican a columnas enteras a la vez, en lugar de recorrer cada valor de una columna. Esto proporciona un código más eficiente y conciso.

Con este sencillo código, puede crear una nueva columna llamada 'total_sales' en el DataFrame. El contenido de esta columna consistirá en la suma de las ventas en las tres regiones, línea por línea.

Podemos aprovechar este método para crear columnas a partir de fórmulas útiles. Por ejemplo, si queremos calcular la parte de las ventas totales que proceden de la UE, podemos hacerlo así:

df = pd.read_csv('/datasets/vg_sales.csv')
# crear la columna total_ventas y rellenarla
df['total_sales'] = df['na_sales'] + df['eu_sales'] + df['jp_sales']

# crear la columna eu_sales_share y rellenarla
df['eu_sales_share'] = df['eu_sales'] / df['total_sales']
print(df['eu_sales_share'].head())

0    0.390876
1    0.090702
2    0.395904
3    0.366533
4    0.292627
Name: eu_sales_share, dtype: float64

## Generar columnas booleanas
Imagina que queremos que una columna indique si algo es verdadero. Podemos crearla mediante los operadores de comparación ==, <, >=, etcétera. Por ejemplo, vamos a crear una columna que comprueba si la empresa distribuidora es Nintendo:

df = pd.read_csv('/datasets/vg_sales.csv')

# crear la columna is_nintendo y rellenarla
df['is_nintendo'] = df['publisher'] == 'Nintendo'
print(df['is_nintendo'].head())

0    True
1    True
2    True
3    True
4    True
Name: is_nintendo, dtype: bool

¡Mira cuánto se parece esto a lo que vimos en las lecciones de filtrado de datos! La mayor parte del filtrado de datos se obtiene aplicando columnas booleanas como una "máscara" sobre los datos. Recuerda que también podemos hacer esto con el conveniente método isin(), que comprueba si un valor está en una lista:

df = pd.read_csv('/datasets/vg_sales.csv')

# asegúrate de que estés comparando minúsculas
print(df['platform'].str.lower().isin(['gb', 'wii']).head())

0     True      esto quiere decir que si esta en la lista
1    False
2     True
3     True
4     True
Name: platform, dtype: bool

Primero, convertimos el contenido de la columna a minúscula para evitar errores como 'wii' versus 'Wii'.

## Columnas categóricas
Trabajar con datos de string sin procesar rara vez nos ayuda para el análisis de datos, pues por lo general las columnas de string necesitan algún tipo de procesamiento.

Si la columna de string representa un conjunto de categorías, es mucho mejor tratar esos valores directamente como categorías.

Convirtiendo una columna en un tipo de dato categórico en lugar de dejarla como cadena, podemos ahorrar memoria y agilizar el análisis, sobre todo en el caso de grandes conjuntos de datos. Dado que las columnas categóricas solo almacenan un único número (el ID de categoría) para cada entrada, en lugar del texto completo de la entrada. Además, el uso de categorías puede facilitar determinados análisis, como la agrupación de datos por categorías o el filtrado de datos en función de varias categorías a la vez. Y esto se puede hacer con el tipo de datos categorical (categórico).

Mira los valores únicos de la columna 'platform' (plataforma).

df = pd.read_csv('/datasets/vg_sales.csv')
print(df['platform'].unique())

['Wii' 'NES' 'GB' 'DS' 'X360' 'PS3' 'PS2' 'SNES' 'GBA' 'PS4' '3DS' 'N64'
 'PS' 'XB' 'PC' '2600' 'PSP' 'XOne' 'WiiU' 'GC' 'GEN' 'DC' 'PSV' 'SAT'
 'SCD' 'WS' 'NG' 'TG16' '3DO' 'GG' 'PCFX']

Podemos convertir 'platform' de columna de string a columna categórica usando el método astype() que aprendiste en el capítulo anterior:

df = pd.read_csv('/datasets/vg_sales.csv')

df['platform'] = df['platform'].astype('category')
print(df['platform'].head())

0    Wii
1    NES
2    Wii
3    Wii
4     GB
Name: platform, dtype: category
Categories (31, object): ['2600', '3DO', '3DS', 'DC', ..., 'WiiU', 'X360', 'XB', 'XOne']

Observa que solo hay 31 categorías a pesar de que hay 16 719 entradas. Cuando la columna está almacenada como strings, necesitamos mantener el texto completo de las 16 719 entradas. Cuando se almacena como category (categoría), solo almacenamos un número: el ID de categoría. ¡Qué bonito!